In [ ]:
#!/usr/bin/env python3
"""
Python script for downloading data from the NASA CMR API.

Fixes:
- Uses earthaccess for authenticated downloads from GES DISC Earthdata Cloud
- Avoids raw requests.get(url) on protected data URLs, which was causing 401 errors
- Keeps the CMR granule search logic
- Fixes Python <3.10 compatibility by removing `str | None`

Requirements:
    pip install tqdm earthaccess requests

To run:
    python download_files_NLDAS_FORB0125_H_2.0.py
"""

import os
import shutil
import requests
import earthaccess
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List, Tuple

CMR_BASE_URL = "https://cmr.earthdata.nasa.gov"

SHORT_NAME = "NLDAS_FORB0125_H"
VERSION = "2.0"
FILTER_TEMPORAL = "2000-01-03T00:00:00.000Z,2000-01-04T23:59:59.000Z"
FILTER_BBOX = ""
FILTER_SEARCH = ""
FILTER_CLOUD_COVER_MIN = ""
FILTER_CLOUD_COVER_MAX = ""
DOWNLOAD_DIR = f"./{SHORT_NAME}_{VERSION}"
MAX_WORKERS = 5  # number of parallel downloads

os.makedirs(DOWNLOAD_DIR, exist_ok=True)


def query_cmr_granules(
    short_name: str,
    version: str,
    page_size: int = 2000,
    search_after: Optional[str] = None,
    **extra_params,
) -> Tuple[requests.Response, List[dict]]:
    """
    Query the CMR granules endpoint.
    """
    url = f"{CMR_BASE_URL}/search/granules.umm_json"

    params = {
        "short_name": short_name,
        "version": version,
        "page_size": page_size,
        **extra_params,
    }

    headers = {"Accept": "application/json"}
    if search_after:
        headers["CMR-Search-After"] = search_after

    response = requests.get(url, params=params, headers=headers, timeout=120)

    try:
        response.raise_for_status()
    except requests.exceptions.HTTPError:
        print("Failed to fetch granules:", response.text)
        raise

    data = response.json()
    items = data.get("items", [])
    return response, items


def find_download_url(item: dict) -> Optional[str]:
    """
    Extract a preferred downloadable URL from a granule item.
    Prefer GET DATA links.
    """
    related_urls = item.get("umm", {}).get("RelatedUrls", [])

    for related_url in related_urls:
        if related_url.get("Type") == "GET DATA" and related_url.get("URL"):
            return related_url["URL"]

    for related_url in related_urls:
        if related_url.get("URL"):
            return related_url["URL"]

    return None


# def login_earthdata():
#     """
#     Log in to Earthdata using earthaccess.
#     This will use existing credentials if already configured,
#     and can prompt interactively if needed.
#     """
#     try:
#         auth = earthaccess.login(strategy="interactive", persist=True)
#         print("Successfully logged into Earthdata.")
#         return auth
#     except Exception as e:
#         raise RuntimeError(
#             "Earthdata login failed. Make sure your Earthdata account is valid "
#             "and authorized for GES DISC."
#         ) from e
AUTH = None

def login_earthdata():
    global AUTH
    if AUTH is None:
        AUTH = earthaccess.login(strategy="interactive", persist=True)
    return AUTH
def download_file(url: str):
    filename = os.path.basename(url)
    local_filename = os.path.join(DOWNLOAD_DIR, filename)

    if os.path.exists(local_filename):
        return "skipped", filename

    try:
        # 🔥 Ensure auth exists in THIS thread
        login_earthdata()

        paths = earthaccess.download(url, local_path=DOWNLOAD_DIR, threads=1)

        if not paths:
            return "failed", filename

        return "success", filename

    except Exception as e:
        print(f"⚠️ Error downloading {filename}: {e}")
        return "failed", filename
    
def download_data_from_cmr(
    short_name: str,
    version: str,
    total_granules: int,
    page_size: int = 2000,
    **params,
):
    """
    Fetch granules from CMR, collect download URLs, then download them.
    """
    search_after_value = None
    all_download_urls: List[str] = []
    granules_without_urls = 0

    print("Collecting download URLs...")
    with tqdm(total=total_granules, desc="Collecting URLs", unit="granule") as pbar:
        while True:
            response, items = query_cmr_granules(
                short_name, version, page_size, search_after_value, **params
            )

            for item in items:
                pbar.update(1)
                download_url = find_download_url(item)

                if download_url:
                    all_download_urls.append(download_url)
                else:
                    granules_without_urls += 1

            search_after_value = response.headers.get("CMR-Search-After")

            if not search_after_value:
                break

    print(f"Found {len(all_download_urls)} files to download")
    if granules_without_urls > 0:
        print(f"⚠️ {granules_without_urls} granules have no download URLs")

    downloaded_count = 0
    skipped_count = 0
    failed_count = 0

    with tqdm(total=len(all_download_urls), desc="Downloading files", unit="file") as pbar:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_url = {
                executor.submit(download_file, url): url for url in all_download_urls
            }

            for future in as_completed(future_to_url):
                status, filename = future.result()

                if status == "success":
                    downloaded_count += 1
                elif status == "skipped":
                    skipped_count += 1
                else:
                    failed_count += 1
                    print(f"Failed: {filename}")

                pbar.update(1)

    print(f"Downloaded: {downloaded_count} files")
    if skipped_count > 0:
        print(f"Skipped: {skipped_count} files (already exist)")
    if failed_count > 0:
        print(f"Failed: {failed_count} files")


def fetch_total_granules_count_from_cmr(short_name: str, version: str, **params) -> int:
    """
    Fetch the total number of matching granules from CMR.
    """
    response, _ = query_cmr_granules(short_name, version, page_size=1, **params)
    data = response.json()
    return data.get("hits", 0)


def main():
    """
    Main entry point.
    """
    print("Logging into Earthdata...")
    login_earthdata()

    filter_params = {}

    if FILTER_TEMPORAL:
        filter_params["temporal"] = FILTER_TEMPORAL

    if FILTER_BBOX:
        filter_params["bounding_box"] = FILTER_BBOX

    if FILTER_SEARCH:
        filter_params["producer_granule_id[]"] = FILTER_SEARCH
        filter_params["options[producer_granule_id][pattern]"] = "true"

    if FILTER_CLOUD_COVER_MIN and FILTER_CLOUD_COVER_MAX:
        filter_params["cloud_cover"] = f"{FILTER_CLOUD_COVER_MIN},{FILTER_CLOUD_COVER_MAX}"

    total_granules = fetch_total_granules_count_from_cmr(
        SHORT_NAME, VERSION, **filter_params
    )
    print(f"Total granules: {total_granules:,}")

    download_data_from_cmr(SHORT_NAME, VERSION, total_granules, **filter_params)

    print("✅ All downloads complete.")


if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
from netCDF4 import Dataset
import pandas as pd
from os import listdir
from os.path import isfile, join

onlyfiles = [f for f in listdir(r'C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\NLDAS_FORB0125_H_2.0') if isfile(join(r'C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\NLDAS_FORB0125_H_2.0', f))]
print(onlyfiles)
num_hours = len(onlyfiles)
for files in onlyfiles:
    ds = Dataset(f'NLDAS_FORB0125_H_2.0\\{files}', mode='r')
    
features_list = ["Rainf", "Wind_E", "Wind_N", "Tair", "Qair"]

# Importing Nodes and Edges of Network
nodes = pd.read_csv(f"Florida_nodeList.csv")
edges = pd.read_csv(f"Florida_edgeList.csv")

lat_vals = ds.variables["lat"][:]
lon_vals = ds.variables["lon"][:]
# Loop through weather events
rainNodes = np.zeros((num_hours, len(nodes.index)))
windNodes = np.zeros((num_hours, len(nodes.index)))
tempNodes = np.zeros((num_hours, len(nodes.index)))
humidityNodes = np.zeros((num_hours, len(nodes.index)))

# LOOP through file
for idx,files in enumerate(onlyfiles):
    print(idx)
    ds = Dataset(f'NLDAS_FORB0125_H_2.0\\{files}', mode='r')
    # Loop through feature
    for feature in features_list:
        # Loop through node
        for i in nodes.index:
            
            # Grab node coordinates
            long = np.round(float(nodes["longitude"][i]),2)
            lat = np.round(float(nodes["latitude"][i]),2)

            # Query NLDAS2 for Weather Data
            lat_idx = np.abs(lat_vals - lat).argmin()
            lon_idx = np.abs(lon_vals - long).argmin()

            rain = np.round(ds.variables["Rainf"][0, lat_idx, lon_idx], 2)
            u = np.round(ds.variables["Wind_E"][0, lat_idx, lon_idx], 2)
            v = np.round(ds.variables["Wind_N"][0, lat_idx, lon_idx], 2)
            temp = np.round(ds.variables["Tair"][0, lat_idx, lon_idx], 2)
            humidity = np.round(ds.variables["Qair"][0, lat_idx, lon_idx], 2)
            # print(f"Node {i} - Rain: {rain}, Wind U: {u}, Wind V: {v}, Temp: {temp}, Humidity: {humidity}")
            # Convert uv wind components to wind speed
            tempWind = np.sqrt(np.square(u) + np.square(v))
            # Append the rain to node event lists
            rainNodes[idx, i] = rain
            windNodes[idx, i] = tempWind * 2.23694
            tempNodes[idx, i] = temp
            humidityNodes[idx, i] = humidity

    # Convert Lists to dataframe
    events = pd.DataFrame(rainNodes)
    events1 = pd.DataFrame(windNodes)
    events2 = pd.DataFrame(tempNodes)
    events3 = pd.DataFrame(humidityNodes)

# Save Dataframes to csv's
pd.DataFrame.to_csv(events, f'./Florida/Rain/nodes/weatherEvent.csv')
pd.DataFrame.to_csv(events1, f'./Florida/Wind/nodes/weatherEvent.csv')
pd.DataFrame.to_csv(events2, f'./Florida/Temperature/nodes/weatherEvent.csv')
pd.DataFrame.to_csv(events3, f'./Florida/Humidity/nodes/weatherEvent.csv')


['NLDAS_FORB0125_H.A20000103.0000.020.nc', 'NLDAS_FORB0125_H.A20000103.0100.020.nc', 'NLDAS_FORB0125_H.A20000103.0200.020.nc', 'NLDAS_FORB0125_H.A20000103.0300.020.nc', 'NLDAS_FORB0125_H.A20000103.0400.020.nc', 'NLDAS_FORB0125_H.A20000103.0500.020.nc', 'NLDAS_FORB0125_H.A20000103.0600.020.nc', 'NLDAS_FORB0125_H.A20000103.0700.020.nc', 'NLDAS_FORB0125_H.A20000103.0800.020.nc', 'NLDAS_FORB0125_H.A20000103.0900.020.nc', 'NLDAS_FORB0125_H.A20000103.1000.020.nc', 'NLDAS_FORB0125_H.A20000103.1100.020.nc', 'NLDAS_FORB0125_H.A20000103.1200.020.nc', 'NLDAS_FORB0125_H.A20000103.1300.020.nc', 'NLDAS_FORB0125_H.A20000103.1400.020.nc', 'NLDAS_FORB0125_H.A20000103.1500.020.nc', 'NLDAS_FORB0125_H.A20000103.1600.020.nc', 'NLDAS_FORB0125_H.A20000103.1700.020.nc', 'NLDAS_FORB0125_H.A20000103.1800.020.nc', 'NLDAS_FORB0125_H.A20000103.1900.020.nc', 'NLDAS_FORB0125_H.A20000103.2000.020.nc', 'NLDAS_FORB0125_H.A20000103.2100.020.nc', 'NLDAS_FORB0125_H.A20000103.2200.020.nc', 'NLDAS_FORB0125_H.A20000103.2300.

C:\Users\ke119419\AppData\Local\Temp\ipykernel_27344\2725113855.py:52: UserWarning: Warning: converting a masked element to nan.
  rainNodes[idx, i] = rain
C:\Users\ke119419\AppData\Local\Temp\ipykernel_27344\2725113855.py:53: UserWarning: Warning: converting a masked element to nan.
  windNodes[idx, i] = tempWind * 2.23694
C:\Users\ke119419\AppData\Local\Temp\ipykernel_27344\2725113855.py:54: UserWarning: Warning: converting a masked element to nan.
  tempNodes[idx, i] = temp
C:\Users\ke119419\AppData\Local\Temp\ipykernel_27344\2725113855.py:55: UserWarning: Warning: converting a masked element to nan.
  humidityNodes[idx, i] = humidity
